[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/09_repertoire/09_repertoire_lab.ipynb)


# 09 — repertoire & naturalness (OAS 직접 다운로드)

> 본문: [`09_repertoire.md`](09_repertoire.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 8초** (실측)

**이 노트북은 도구를 직접 돌립니다.** **진짜 OAS data unit** 을 직접 내려받아 CDR3 길이 분포를 만들고, 후보 항체의 위치를 재요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "09_repertoire"
PIP_PKGS = "pandas matplotlib anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

if shutil.which("hmmscan") is None:      # ANARCI 가 부르는 실행파일 — pip 로는 안 깔려요
    if IN_COLAB:
        _APT.append("hmmer")
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")           # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨집니다

if _APT:
    _run("apt-get -qq update")           # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("apt-get -qq install -y " + " ".join(_APT))

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — OAS data unit 다운로드 + CDR3 길이 집계 (본문 9.1)

```bash
python scripts/fetch_oas_unit.py --out my_run/oas_subset.tsv.gz
python scripts/oas_cdr3_length.py my_run/oas_subset.tsv.gz --column cdr3_aa \
    --out my_run/oas_cdr3_length_summary.csv
```
받는 unit — **Eliyahu et al. 2018 · human PBMC · heavy IgM · run ERR2843400** (productive 17,807 서열).
OAS 원본 파일은 **첫 줄이 메타데이터(JSON)** 라 그냥 읽으면 컬럼을 못 찾아요 — 스크립트가 자동 처리합니다.

> **이 unit 의 한계** (본문 9.1) — HCV 코호트 **피험자 1명(subject CI15)의 IgM 레퍼토리**예요.
> "인간 레퍼토리 일반"의 대표값이 아니라 한 사람의 한 시점 샘플입니다.
> 실전 benchmark 라면 여러 건강 도너의 여러 unit 을 합치세요(같은 스크립트에 `--url` 만 바꾸면 됩니다).

In [ ]:
run_tool([PY, SCRIPTS/"fetch_oas_unit.py", "--out", "my_run/oas_subset.tsv.gz"])
run_tool([PY, SCRIPTS/"oas_cdr3_length.py", resolve("oas_subset.tsv.gz"),
          "--column", "cdr3_aa", "--out", "my_run/oas_cdr3_length_summary.csv"])

## 2) 내 결과 확인 — 후보의 CDR-H3 를 IMGT 로 재기 (본문 9.2)

이 unit 은 **heavy** 서열이라 비교 대상도 heavy 여야 해요. FASTA 의 첫 레코드가 중쇄라는 보장은 없으니
**전 레코드를 numbering 해서 `chain_type` 을 확인**하고 H 사슬만 분포에 얹습니다.
길이 정의도 맞춰야 해요 — OAS `cdr3_aa` 는 IMGT CDR3(105–117)예요.

In [ ]:
from abnumber import Chain

def read_fasta(path):
    d, n = {}, None
    for line in open(path):
        line = line.strip()
        if line.startswith(">"): n = line[1:].split()[0]; d[n] = ""
        elif n: d[n] += line
    return d

seqs, chains = read_fasta("data/demo_mab.fa"), {}
for sid, seq in seqs.items():
    try:
        chains[sid] = Chain(seq, scheme="imgt")
    except Exception as e:
        print(f"{sid}: IMGT numbering 실패 — {type(e).__name__} "
              "(ANARCI 는 hmmscan 실행파일이 필요해요 — 부트스트랩 로그 확인)")
for sid, ch in chains.items():
    print(f"{sid}: chain_type={ch.chain_type} | CDR3(IMGT) {ch.cdr3_seq} = {len(ch.cdr3_seq)} aa")

heavy = [(sid, ch) for sid, ch in chains.items() if ch.chain_type == "H"]
light = [(sid, ch) for sid, ch in chains.items() if ch.chain_type != "H"]
cand = cand_id = None
cdr3_by_scheme = {}

if heavy:
    cand_id, hch = heavy[0]
    cand = len(hch.cdr3_seq)
    if len(heavy) > 1:
        print("H 사슬이", len(heavy), "개예요 — 첫 번째", cand_id, "를 대표로 씁니다")
    print(f"\n비교 대상 — {cand_id} 의 CDR-H3 {cand} aa (IMGT)")
    if light:
        print("경쇄", ", ".join(f"{s}({ch.chain_type}, CDR-L3 {len(ch.cdr3_seq)} aa)" for s, ch in light),
              "— 이 분포는 heavy 전용이라 경쇄 길이는 여기에 얹지 않아요")
    for scheme in ("imgt", "kabat", "chothia"):
        try:
            cdr3_by_scheme[scheme] = len(Chain(seqs[cand_id], scheme=scheme).cdr3_seq)
        except Exception as e:
            print(f"  {scheme} numbering 실패 — {type(e).__name__}")
    print("scheme 별 CDR-H3 길이 —",
          " · ".join(f"{k} {v} aa" for k, v in cdr3_by_scheme.items()))
    print("판정 — percentile 은 IMGT 값으로만 계산해요. "
          "OAS cdr3_aa 가 IMGT 정의라, 다른 scheme 길이를 얹으면 엉뚱한 percentile 이 나옵니다 (3절에서 실측).")
else:
    print("\nheavy(H) 사슬을 못 찾았어요 — 이 unit 은 heavy 라 percentile 비교를 건너뜁니다.")
    print("경쇄만 있는 FASTA 라면 light data unit(예: ERR..._Light_IGKC.csv.gz)을 "
          "fetch_oas_unit.py --url 로 받아 같은 절차를 돌리세요.")

## 3) 내 결과 확인 — 분포 통계와 후보의 위치 (본문 9.2)

In [ ]:
import pandas as pd
s = pd.read_csv(resolve("oas_cdr3_length_summary.csv"))
need = [x for x in ("cdr3_len", "count") if x not in s.columns]
assert not need, f"집계 CSV 에 {need} 컬럼이 없어요 → oas_cdr3_length.py 를 다시 돌리세요"
s = s.sort_values("cdr3_len").reset_index(drop=True)

n    = int(s["count"].sum())
mean = float((s["cdr3_len"] * s["count"]).sum() / n)
mode = s.loc[s["count"].idxmax()]
cum  = s["count"].cumsum() / n
p05  = int(s.loc[cum >= 0.05, "cdr3_len"].iloc[0])
p95  = int(s.loc[cum >= 0.95, "cdr3_len"].iloc[0])

def pctile(length):
    """이 분포에서 length 이하가 차지하는 비율(%) — 하위 percentile."""
    return 100 * float(s.loc[s["cdr3_len"] <= length, "count"].sum()) / n

print(f"n={n:,} 서열 | 평균 {mean:.2f} aa | 최빈 {int(mode['cdr3_len'])} aa "
      f"({int(mode['count']):,}건, {100*int(mode['count'])/n:.1f}%) | "
      f"범위 {int(s['cdr3_len'].min())}–{int(s['cdr3_len'].max())} aa")
print(f"5–95 percentile 구간 = {p05}–{p95} aa")

tail = s[s["cdr3_len"] <= 4]
if len(tail):
    tn = int(tail["count"].sum())
    print(f"\n왼쪽 꼬리 {int(tail['cdr3_len'].min())}–{int(tail['cdr3_len'].max())} aa 가 "
          f"{tn}건 ({100*tn/n:.2f}%) — 실제 항체보다 시퀀싱/어노테이션 노이즈로 보는 게 맞아요.")
    print("이 집계는 자르지 않고 그대로 뒀어요(최솟값이 그래서 작게 찍힙니다). "
          "자르든 남기든 어느 쪽을 골랐는지 보고서에 밝히는 게 repertoire 분석의 기본기예요.")

if cand is None:
    print("\n후보 heavy 사슬이 없어 percentile 은 계산하지 않습니다.")
else:
    pc = pctile(cand)
    print(f"\n후보 {cand_id} CDR-H3 {cand} aa → 하위 {pc:.1f} percentile")
    if p05 <= cand <= p95:
        print(f"판정 — {p05}–{p95} aa(5–95 percentile) 안이라 길이 측면에서는 자연 human heavy 분포의 정상 범위예요.")
    elif cand > p95:
        print(f"판정 — 95 percentile({p95} aa)보다 길어요. 긴 CDR3 는 발현·안정성·면역원성을 따로 점검하세요.")
    else:
        print(f"판정 — 5 percentile({p05} aa)보다 짧아요. 결합 표면이 좁을 수 있어 affinity·특이성과 함께 보세요.")
    d_mode = abs(cand - int(mode["cdr3_len"]))
    print(f"최빈값 {int(mode['cdr3_len'])} aa 와 {d_mode} aa 차이 · 평균 {mean:.2f} aa 와 {abs(cand-mean):.1f} aa 차이")
    for scheme, L in cdr3_by_scheme.items():
        if scheme != "imgt":
            print(f"  참고) {scheme} 로 재면 {L} aa → 하위 {pctile(L):.1f} percentile "
                  f"({pctile(L) - pc:+.1f} p 이동, 같은 항체인데)")
    print("  그래서 본문 9.2 의 주의대로 후보도 OAS 와 같은 IMGT 정의로 재야 해요.")

## 4) 내 결과 확인 — CDR3 분포 그래프 (본문 9.2)

In [ ]:
from antibody_viz import cdr3_length_distribution
from IPython.display import Image, display

png = "my_run/09_cdr3_length.png"
cdr3_length_distribution(resolve("oas_cdr3_length_summary.csv"),
    "OAS (Eliyahu 2018, human IgM heavy) — CDR3 length distribution",
    png, highlight_len=cand,
    highlight_label=(f"{cand_id} CDR-H3" if cand_id else "candidate"))
display(Image(png))

print(f"파란 막대 = 길이별 서열 수(봉우리 {int(mode['cdr3_len'])} aa) · "
      f"빨간 점선 = 평균 {mean:.2f} aa · 오른쪽 꼬리는 {int(s['cdr3_len'].max())} aa 까지")
if cand is None:
    print("판정 — 후보 heavy 가 없어 주황 실선은 그려지지 않았어요.")
else:
    side = "왼쪽" if cand < int(mode["cdr3_len"]) else ("오른쪽" if cand > int(mode["cdr3_len"]) else "같은 칸")
    print(f"주황 실선 = 후보 {cand} aa — 봉우리의 {side}, 하위 {pctile(cand):.1f} percentile 자리예요.")
    print("판정 — " + ("주황선이 종형의 몸통 안이면 길이 naturalness 는 통과."
                     if p05 <= cand <= p95 else
                     "주황선이 꼬리 쪽이면 왜 그 길이인지 설명할 수 있어야 해요.")
          + " 단, 중앙이라고 좋은 항체라는 뜻은 아니에요 (본문 9.3).")

## 5) 레퍼런스 대조 — 커밋된 OAS 서브셋과 같은가

`data/oas_subset.tsv.gz` 는 **같은 OAS data unit 을 2026-07-14 에 받아 커밋해 둔 실제 데이터**예요
(합성 데이터 아님). 내가 방금 받은 것과 집계가 같아야 정상입니다.

In [ ]:
import pandas as pd, pathlib
ref = pd.read_csv("data/oas_cdr3_length_summary.csv")
mine_p = pathlib.Path("my_run/oas_cdr3_length_summary.csv")
if not mine_p.exists():
    print("my_run 집계가 없어 대조를 건너뜁니다 → 1절 실행 로그(네트워크)를 확인하세요.")
else:
    mine = pd.read_csv(mine_p)
    common = [x for x in ref.columns if x in mine.columns]
    same = mine[common].reset_index(drop=True).equals(ref[common].reset_index(drop=True))
    print(f"내 집계 n={int(mine['count'].sum()):,} / 레퍼런스 n={int(ref['count'].sum()):,} | 완전 일치 = {same}")
    if not same:
        m = mine.merge(ref, on="cdr3_len", how="outer", suffixes=("_mine", "_ref")).fillna(0)
        print(m[m["count_mine"] != m["count_ref"]].head(10).to_string(index=False))
    print("판정 — " + ("같은 unit 을 같은 스크립트로 집계했다는 확인이에요."
                     if same else
                     "OAS 가 data unit 을 갱신했거나 --url 을 바꿔 받았다면 달라지는 게 정상이에요. "
                     "보고서에는 받은 날짜와 unit 이름을 함께 적으세요."))

## 6) V/J germline usage — "목록에 없다"가 뜻하는 것 (본문 9.3)

naturalness 의 두 번째 축은 **germline usage** 예요. 내가 받은 unit 에서 어떤 IGHV/IGHJ 가 흔한지 세고,
후보의 germline 이 그 안에 있는지 봅니다. **없을 때 그 이유가 "희귀"인지 "종이 다름"인지 가르는 게 핵심**이에요.

In [ ]:
import csv, pandas as pd
rep = pd.read_csv(resolve("oas_subset.tsv.gz"), sep="\t")
print(f"unit 서열 {len(rep):,}개 | 컬럼 {list(rep.columns)}")

usage = {}
for col, label in [("v_call", "IGHV"), ("j_call", "IGHJ")]:
    if col not in rep.columns:
        print(f"[{label}] {col} 컬럼이 없어 건너뜁니다"); continue
    g = rep[col].astype(str).str.split("*").str[0]
    usage[col] = g
    top = g.value_counts()
    print(f"\n[{label} top5 / 이 unit 의 {g.nunique()}종]")
    for name, k in top.head(5).items():
        print(f"   {name:12s} {k:6,d}  ({100*k/len(rep):.1f}%)")

# --- 후보의 germline — abnumber(ANARCI) 로 직접, 실패하면 Ch.04 결과로 ---------
v_gene = j_gene = species = None
if cand_id:
    try:
        hv = Chain(seqs[cand_id], scheme="imgt", assign_germline=True)
        v_gene, j_gene, species = hv.v_gene, hv.j_gene, hv.species
        print("\n[germline 출처] abnumber/ANARCI 직접 할당")
    except Exception as e:
        print("\nabnumber germline 할당 실패 —", type(e).__name__, "→ Ch.04 numbering CSV 로 대체합니다")
    if v_gene is None:
        for sub in ("my_run", "data"):
            p = ADV_ROOT / "04_numbering" / sub / "demo_imgt_H.csv"
            if not p.exists():
                continue
            for row in csv.DictReader(open(p)):
                if row.get("Id") == cand_id and (row.get("chain_type") or "").strip() == "H":
                    v_gene, j_gene = row.get("v_gene"), row.get("j_gene")
                    species = (row.get("identity_species") or row.get("hmm_species") or "").strip()
            if v_gene:
                print("[germline 출처]", p); break

if not v_gene:
    print("\n후보 germline 을 못 읽어 usage 비교를 건너뜁니다 "
          "(Ch.04 노트북을 같은 FASTA 로 먼저 돌리면 채워집니다).")
elif "v_call" not in usage:
    print("\nunit 에 v_call 이 없어 usage 비교를 건너뜁니다.")
else:
    v_fam = str(v_gene).split("*")[0]
    genes = sorted(set(usage["v_call"]))
    hit   = int((usage["v_call"] == v_fam).sum())
    print(f"\n후보 heavy germline — V {v_gene} / J {j_gene} / ANARCI species {species or '미상'}")
    print(f"이 unit 에서 {v_fam} 사용 = {hit}건 (unit 의 IGHV {len(genes)}종 중)")
    if hit:
        rank = int((usage["v_call"].value_counts() > hit).sum()) + 1
        print(f"판정 — 목록 안에 있어요. usage {100*hit/len(rep):.2f}% 로 {len(genes)}종 중 {rank}위 "
              "→ 이 도너에게 흔한 germline 이에요.")
    else:
        fams = sorted({g.split("-")[0] for g in genes})
        print(f"이 unit 의 IGHV family 는 {', '.join(fams)} 뿐이고, 후보의 {v_fam.split('-')[0]} 은 그 안에 없어요.")
        if species and str(species).lower() != "human":
            print(f"판정 — 0건인 이유는 '희귀 germline' 이 아니라 **종이 다르기 때문**이에요. "
                  f"후보 heavy 는 {species} germline 으로 맞았고, 이 unit 은 human repertoire 라 "
                  f"{v_fam} 이라는 gene 자체가 없습니다.")
            print("판정 — 종이 다른 germline 은 human usage percentile 의 비교 대상이 아니에요. "
                  "여기서 읽을 것은 usage 순위가 아니라 humanization 이 필요하다는 사실(Ch.04·Ch.05)입니다.")
        else:
            print("판정 — 같은 종인데 0건이면 그때가 '이 unit 기준 희귀' 예요. "
                  "다만 unit 1개(도너 1명)라 희귀 판정은 여러 unit 을 합쳐서 해야 합니다.")
    if j_gene and "j_call" in usage:
        j_fam = str(j_gene).split("*")[0]
        jhit = int((usage["j_call"] == j_fam).sum())
        if jhit:
            print(f"참고) J 이름 {j_fam} 은 이 human unit 에도 {jhit:,}건 있어요 — "
                  "gene 이름은 종을 넘어 겹치니 이름이 같다고 같은 germline 은 아니에요.")

print("\n본문 9.3 원칙 — 흔하다고 좋은 항체가 아니고, 드물다고 나쁜 후보도 아니에요. "
      "다만 '왜 드문지' 설명할 수 있어야 하고, naturalness 는 단독 판정에 쓰지 않아요.")

## 7) 통합 triage — 전 챕터 결과를 한 표로 (본문 9.4)

이 가이드의 **결론 산출물**이에요. 축별로 각 챕터의 `my_run/` 을 먼저 찾고 없으면 그 챕터의 커밋본을 읽어
실측과 판정을 한 표로 모읍니다. 판정은 전부 **읽은 값으로 계산**해요 — 여러분 서열로 바꿔 돌리면 판정도 바뀝니다.
표는 `my_run/triage_summary.csv` 로도 저장돼 보고서에 그대로 붙일 수 있어요.

In [ ]:
import csv, pathlib, pandas as pd

def ch_file(chapter, name):
    """다른 챕터의 산출물 — 그 챕터 my_run/ 을 먼저, 없으면 커밋된 data/ 를 쓴다."""
    for sub in ("my_run", "data"):
        p = ADV_ROOT / chapter / sub / name
        if p.exists():
            return p, sub
    return None, None

rows = []
def add(axis, chapter, value, verdict, src):
    rows.append({"축": axis, "챕터": chapter, "실측": value, "판정": verdict, "출처": src})

# --- 04 numbering·germline -------------------------------------------------
germ, src04 = [], set()
for fn in ("demo_imgt_H.csv", "demo_imgt_KL.csv"):
    p, sub = ch_file("04_numbering", fn)
    if not p:
        continue
    src04.add(sub)
    for r in csv.DictReader(open(p)):
        germ.append((r.get("Id"), r.get("chain_type"), r.get("v_gene"),
                     (r.get("identity_species") or r.get("hmm_species") or "").strip()))
if germ:
    nonhuman = [g for g in germ if g[3] and g[3].lower() != "human"]
    add("numbering·germline", "04",
        " / ".join(f"{i}({t}) {v} [{sp or '?'}]" for i, t, v, sp in germ),
        (f"주의 — 비human germline {len(nonhuman)}개 사슬 → humanization 검토" if nonhuman
         else "양호 — 전 사슬 human germline"),
        "+".join(sorted(src04)))

# --- 05 humanness ----------------------------------------------------------
p, sub = ch_file("05_humanness", "demo_sapiens_scores.csv")
if p:
    d = pd.read_csv(p)
    if {"chain", "input_aa"} <= set(d.columns):
        aa = list("ACDEFGHIKLMNPQRSTVWY")
        d["p"] = [r[r["input_aa"]] if r["input_aa"] in aa else None for _, r in d.iterrows()]
        hum = d.groupby("chain")["p"].mean()
        val = " / ".join(f"{k} {v:.3f}" for k, v in hum.items())
        po, so = ch_file("05_humanness", "demo_mab.fa")
        ph, sh = ch_file("05_humanness", "demo_humanized.fa")
        if po and ph:
            o, h = read_fasta(po), read_fasta(ph)
            muts = {k: sum(1 for x, y in zip(o[k], h[k]) if x != y)
                    for k in o if k in h and len(o[k]) == len(h[k])}
            if muts:
                val += " · humanizing 변이 " + " / ".join(f"{k} {v}" for k, v in muts.items())
        worst = float(hum.min())
        add("humanness", "05", val,
            ("양호 — 전 사슬 0.8 이상" if worst >= 0.8 else f"주의 — 최저 {worst:.3f} (기준 0.8 미만)"),
            sub)

# --- 06 구조 신뢰도 ---------------------------------------------------------
p, sub = ch_file("06_structure", "demo_antibody_igfold.pdb")
if p:
    prof = {}
    for line in open(p):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            prof.setdefault(line[21], []).append((int(line[22:26]), float(line[60:66])))
    if prof:
        stat = {k: (sum(b for _, b in v) / len(v), max(v, key=lambda t: t[1]))
                for k, v in sorted(prof.items())}
        val = " / ".join(f"{k} mean {m:.3f} Å, max {mx[1]:.2f} Å (res {mx[0]})"
                         for k, (m, mx) in stat.items())
        worst_mean = max(m for m, _ in stat.values())
        worst_max  = max(mx[1] for _, mx in stat.values())
        verdict = ("양호" if worst_mean < 1.0 else "주의") + f" — 사슬 평균 최대 {worst_mean:.3f} Å"
        if worst_max >= 2.0:
            verdict += f", 최대오차 {worst_max:.2f} Å 잔기(loop)는 따로 확인"
        add("구조 신뢰도", "06", val, verdict, sub)

# --- 07 interface (예시 복합체) ---------------------------------------------
p, sub = ch_file("07_interface", "contacts_H_N.tsv")
if p:
    pairs = []
    for line in open(p):
        if "atom_contacts=" in line:
            left, k = line.rstrip().split("atom_contacts=")
            a, b = left.rstrip("\t").split("\t")
            pairs.append((a.strip(), b.strip(), int(k)))
    if pairs:
        add("interface", "07",
            f"{len(pairs)} residue pair · paratope {len({a for a, _, _ in pairs})} · "
            f"epitope {len({b for _, b, _ in pairs})} · 원자접촉 {sum(k for *_, k in pairs)}",
            "참고 — 후보 자신의 복합체가 아니라 예시 구조(1A14) 기준이에요",
            sub)

# --- 08 developability -----------------------------------------------------
p, sub = ch_file("08_developability", "liability.csv")
if p:
    d = pd.read_csv(p)
    mcols = [x for x in ("N_glycosylation_NXS_T", "deamidation_NG",
                         "deamidation_NS", "isomerization_DG") if x in d.columns]
    tot = sum(1 for x in mcols for v in d[x] for q in str(v).split(";") if q.strip().isdigit())
    odd = int(sum(1 for v in d.get("cysteine_count", []) if int(v) % 2 == 1))
    flags = ([f"unpaired Cys {odd}사슬"] if odd else []) + ([f"motif {tot}건"] if tot else [])
    add("developability", "08", f"홀수 Cys 사슬 {odd}개 · liability motif {tot}건",
        ("양호 — 서열 liability 깨끗" if not flags else "주의 — " + ", ".join(flags)), sub)

# --- 09 naturalness --------------------------------------------------------
_cand, _pct = globals().get("cand"), globals().get("pctile")
if _cand is not None and _pct is not None:
    add("naturalness", "09",
        f"CDR-H3 {_cand} aa (IMGT) · 하위 {_pct(_cand):.1f} percentile · n={n:,}",
        (f"양호 — 5–95 percentile({p05}–{p95} aa) 안" if p05 <= _cand <= p95
         else f"주의 — 5–95 percentile({p05}–{p95} aa) 밖"),
        "이 챕터")

tri = pd.DataFrame(rows)
display(tri)
out = pathlib.Path("my_run/triage_summary.csv")
tri.to_csv(out, index=False, encoding="utf-8-sig")
print("저장:", out)

warn = [r["축"] for r in rows if str(r["판정"]).startswith("주의")]
print(f"\n축 {len(rows)}개 중 주의 {len(warn)}개" + (" — " + ", ".join(warn) if warn else ""))
print("판정 — " + (", ".join(warn) + " 축이 남은 과제예요. 나머지 축은 기준을 통과했습니다."
                 if warn else "모든 축이 기준을 통과했어요."))
print("이 표가 보고서의 첫 페이지예요 — 축·수치·판정·출처를 한 줄씩 (본문 9.4 · Ch.10 체크리스트).")

> 다음 → 본문 [10. 부록](../10_appendix/10_appendix.md)